# FHIR R4 Terminology Server Demo

This notebook demonstrates the medterm4ds FHIR R4 terminology server — a
UMLS-backed terminology service supporting code lookup, validation,
cross-system mapping, hierarchy checking, ValueSet expansion, and
text-to-code search (lexical + semantic).

**Prerequisites**: `pip install medterm4ds[fhir]` and a UMLS DuckDB at
`MEDTERM4DS_DB`.

## Start the server

In [17]:
import os
os.environ.setdefault('MEDTERM4DS_DB', '/mnt/d/medterm4ds/data/umls_current.duckdb')
os.environ.setdefault('MEDTERM4DS_FHIR4PX_BASELINE', '/mnt/d/medterm4ds/reports/fhir4px')
os.environ.setdefault('MEDTERM4DS_SEARCH_INDEX_DIR', '/mnt/d/fhir4px-model/dist/naming_bm25')

from starlette.testclient import TestClient
from medterm4ds.apps.fhir_api import FhirApiSettings, create_fhir_app

settings = FhirApiSettings.from_env()
app = create_fhir_app(settings)
client = TestClient(app)
client.__enter__()  # Keep the server open for all cells (call __exit__ in the last cell)

# Test
resp = client.get('/fhir/metadata')
print(f'Server status: {resp.status_code}')
print(f'FHIR version: {resp.json()["fhirVersion"]}')
ops = [op['name'] for r in resp.json()['rest'] for res in r['resource'] for op in res.get('operation', [])]
print(f'Operations: {sorted(ops)}')

Server status: 200
FHIR version: 4.0.1
Operations: ['closure', 'expand', 'lookup', 'search', 'subsumes', 'translate', 'validate-code']


---
## 1. $lookup — Code Details

Given a code, return its display name and custom properties (patient-friendly
name, canonical ICD-10 code, term type).

In [18]:
import json

def show_params(resp, title=''):
    """Pretty-print a FHIR Parameters response."""
    body = resp.json()
    if title:
        print(f'\n=== {title} ===')
    print(f'Status: {resp.status_code} | Resource: {body["resourceType"]}')
    for p in body.get('parameter', []):
        name = p['name']
        if 'part' in p:
            # Property entry: code + value
            parts = {pt['name']: pt.get('valueString', pt.get('valueCode', '')) for pt in p['part']}
            print(f'  {name}: {parts.get("code", "?")} = {parts.get("value", "?")}')
        else:
            val = p.get('valueString', p.get('valueUri', p.get('valueCode', p.get('valueBoolean', '?'))))
            print(f'  {name}: {val}')

# Lookup a SNOMED code
resp = client.get('/fhir/CodeSystem/$lookup', params={
    'system': 'http://snomed.info/sct',
    'code': '44054006',
})
show_params(resp, 'SNOMED 44054006 (Type 2 diabetes)')


=== SNOMED 44054006 (Type 2 diabetes) ===
Status: 200 | Resource: Parameters
  name: SNOMED Clinical Terms (US)
  code: 44054006
  system: http://snomed.info/sct
  display: Type 2 diabetes mellitus
  abstract: False
  property: cui = C0011860
  property: tty = PT
  property: aui = A23027694
  property: patient-friendly = Diabetes Type 2
  property: match-type = same_cui
  property: canonical-code = E11
  property: canonical-system = icd10


In [19]:
# Lookup a RxNorm code (shows tty property)
resp = client.get('/fhir/CodeSystem/$lookup', params={
    'system': 'http://www.nlm.nih.gov/research/umls/rxnorm',
    'code': '860975',
})
show_params(resp, 'RxNorm 860975 (Metformin)')


=== RxNorm 860975 (Metformin) ===
Status: 200 | Resource: Parameters
  name: RxNorm
  code: 860975
  system: http://www.nlm.nih.gov/research/umls/rxnorm
  display: 24 HR metformin hydrochloride 500 MG Extended Release Oral Tablet
  abstract: False
  property: cui = C0978484
  property: tty = SCD
  property: aui = A31696619
  property: patient-friendly = Metformin Oral Product
  property: match-type = group
  property: canonical-code = 860975
  property: canonical-system = rxnorm
  property: tty = SCD


---
## 2. $validate-code — Code Validation

Check if a code exists in a code system.

In [20]:
# Valid code
resp = client.get('/fhir/CodeSystem/$validate-code', params={
    'system': 'http://snomed.info/sct', 'code': '44054006',
})
result = [p for p in resp.json()['parameter'] if p['name'] == 'result'][0]
print(f'44054006 valid: {result["valueBoolean"]}')  # True

# Invalid code
resp = client.get('/fhir/CodeSystem/$validate-code', params={
    'system': 'http://snomed.info/sct', 'code': 'FAKE999',
})
result = [p for p in resp.json()['parameter'] if p['name'] == 'result'][0]
print(f'FAKE999 valid: {result["valueBoolean"]}')  # False

44054006 valid: True
FAKE999 valid: False


---
## 3. $translate — Cross-System Mapping

Map a code from one system to another (e.g., SNOMED → ICD-10).

In [21]:
resp = client.get('/fhir/ConceptMap/$translate', params={
    'system': 'http://snomed.info/sct',
    'code': '44054006',
    'targetsystem': 'http://hl7.org/fhir/sid/icd-10-cm',
})
body = resp.json()
result = [p for p in body['parameter'] if p['name'] == 'result'][0]
print(f'Translate result: {result["valueBoolean"]}')
for p in body['parameter']:
    if p['name'] == 'match':
        parts = {pt['name']: pt for pt in p['part']}
        concept = parts.get('concept', {}).get('valueCoding', {})
        equiv = parts.get('equivalence', {}).get('valueCode', '?')
        print(f'  → {concept.get("code", "?")} ({concept.get("display", "?")}) [{equiv}]')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Translate result: True
  → E11 (Type 2 diabetes mellitus) [equivalent]


### Hierarchy-Walking Mapping (ICD-10 → SNOMED via parent traversal)

Real-world scenario: a FHIR Condition arrives with ICD-10 code **E11.65**
("Type 2 diabetes mellitus with hyperosmolarity"). This specific code doesn't
share a concept (CUI) with any SNOMED code. But its parent **E11** does.

The medterm4ds mapping engine walks up the ICD-10 hierarchy until it finds a
code that shares a CUI with the target system, then reports the match with
the depth and match_type so the caller knows it's a broader approximation.

This is a common pattern when mapping between systems with different
granularity (ICD-10 is very specific; SNOMED may not have an equivalent
at the same level of detail).

In [22]:
# The FHIR $translate uses max_depth=0 (direct same-CUI only).
# For hierarchy-walking mappings, use the medterm4ds Python API directly.
# This demonstrates "broader mapping": a specific ICD-10 code that doesn't
# share a CUI with any SNOMED code, but its parent category does.

from medterm4ds.core.models import CodeRef
from medterm4ds.engines.duckdb import LocalDuckDBEngine
from medterm4ds.services.mapping import get_code_mappings

# Connect directly to the UMLS DB
import duckdb
umls_con = duckdb.connect(
    os.environ.get('MEDTERM4DS_DB', '/mnt/d/medterm4ds/data/umls_current.duckdb'),
    read_only=True,
)
umls_engine = LocalDuckDBEngine(umls_con)

# E11.65 = "Type 2 diabetes mellitus with hyperosmolarity"
# This specific code has NO direct CUI match to any SNOMED code.
# But its parent E11 ("Type 2 diabetes mellitus") shares CUI C0011860
# with SNOMED 44054006 ("Type 2 diabetes mellitus").

print("=== Direct mapping (max_depth=0) ===")
direct = get_code_mappings(
    [CodeRef("ICD10CM", "E11.65")],
    engine=umls_engine,
    target_sources=["SNOMEDCT_US"],
    max_depth=0,
)
print(f"E11.65 → SNOMED (direct): {len(direct)} matches")

print("\n=== Hierarchy-walking mapping (max_depth=3) ===")
with_walk = get_code_mappings(
    [CodeRef("ICD10CM", "E11.65")],
    engine=umls_engine,
    target_sources=["SNOMEDCT_US"],
    max_depth=3,
)
print(f"E11.65 → SNOMED (with walk): {len(with_walk)} matches")
print(f"\nHow it works: E11.65 → walk up to E11 → same-CUI → SNOMED")
print()
for m in with_walk[:5]:
    arrow = "→" if m.match_depth == 0 else f"→ (via depth {m.match_depth})"
    print(f"  {arrow} SNOMED {m.target.code:12s} ({m.target_display or '?'})")
    print(f"      match_type: {m.match_type}")

umls_con.close()

=== Direct mapping (max_depth=0) ===
E11.65 → SNOMED (direct): 0 matches

=== Hierarchy-walking mapping (max_depth=3) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

E11.65 → SNOMED (with walk): 10 matches

How it works: E11.65 → walk up to E11 → same-CUI → SNOMED

  → (via depth 1) SNOMED 190419001    (Diabetes mellitus, adult onset, with other specified manifestation)
      match_type: source_ancestor_same_cui
  → (via depth 2) SNOMED 154672006    (Diabetes mellitus: [adult onset] or [noninsulin dependent])
      match_type: source_ancestor_same_cui
  → (via depth 2) SNOMED 190323008    (Maturity onset diabetes)
      match_type: source_ancestor_same_cui
  → (via depth 2) SNOMED 190384004    (Type II diabetes mellitus)
      match_type: source_ancestor_same_cui
  → (via depth 2) SNOMED 267468009    (Diabetes mellitus -adult onset)
      match_type: source_ancestor_same_cui


---
## 4. $subsumes — Hierarchy Checking

Check if one code is an ancestor of another (subsumption).

In [23]:
cases = [
    ('73211009', '44054006', 'Diabetes → Type 2 diabetes'),
    ('44054006', '73211009', 'Type 2 diabetes → Diabetes'),
    ('44054006', '44054006', 'Same code'),
    ('44054006', '860975',  'Diabetes → Metformin'),
]
for code_a, code_b, desc in cases:
    resp = client.get('/fhir/CodeSystem/$subsumes', params={
        'system': 'http://snomed.info/sct',
        'codeA': code_a,
        'codeB': code_b,
    })
    outcome = [p for p in resp.json()['parameter'] if p['name'] == 'outcome'][0]
    print(f'{desc:45s} → {outcome["valueCode"]}')

Diabetes → Type 2 diabetes                    → subsumes
Type 2 diabetes → Diabetes                    → subsumed-by
Same code                                     → equivalent
Diabetes → Metformin                          → not-subsumed


---
## 5. $expand — ValueSet Expansion

Expand a ValueSet by text filter (EHR autocomplete) or intensional definition
(all descendants of X).

In [24]:
# Text filter: "diabetes"
resp = client.get('/fhir/ValueSet/$expand', params={'filter': 'diabetes', 'count': 5})
body = resp.json()
contains = body['expansion']['contains']
print(f'Filter "diabetes": {body["expansion"]["total"]} matches')
for c in contains:
    print(f'  {c["system"].split("/")[-1]:15s} {c["code"]:15s} {c.get("display", "")}')

Filter "diabetes": 5 matches
  loinc.org       LA10529-8       Diabetes
  loinc.org       LP128793-9      Diabetes
  loinc.org       MTHU040702      Diabetes
  icd-10-cm       E11             diabetes NOS
  loinc.org       LP417553-7      Diabetes risk


In [25]:
# Intensional: all descendants of SNOMED 73211009 (Diabetes mellitus)
resp = client.post('/fhir/ValueSet/$expand', json={
    'resourceType': 'ValueSet',
    'compose': {
        'include': [{
            'system': 'http://snomed.info/sct',
            'filter': [{'property': 'concept', 'op': 'is-a', 'value': '73211009'}],
        }],
    },
})
body = resp.json()
print(f'Intensional is-a 73211009: {body["expansion"]["total"]} codes')
for c in body['expansion']['contains'][:5]:
    print(f'  {c["code"]:15s} {c.get("display", "")}')
if body['expansion']['total'] > 5:
    print(f'  ... and {body["expansion"]["total"] - 5} more')

Intensional is-a 73211009: 762 codes
  73211009        Diabetes mellitus
  105401000119101 Diabetes mellitus due to pancreatic injury
  111552007       Diabetes mellitus without complication
  11530004        Brittle diabetes mellitus
  11687002        Gestational diabetes mellitus
  ... and 757 more


---
## 6. $search — Text-to-Code Search

Three modes:
- **lexical**: BM25 token matching (~1ms, covers 80-90% of queries)
- **semantic**: SapBERT embedding + FAISS ANN (~100ms, catches novel phrasings)
- **hybrid**: BM25 retrieve + SapBERT re-rank (~110ms, best accuracy)

Each result has a match-grade: `certain`, `probable`, or `possible`.

In [26]:
def show_search(query, mode='lexical', system=None):
    params = {'query': query, 'count': 5, 'searchMode': mode}
    if system:
        params['system'] = system
    resp = client.get('/fhir/CodeSystem/$search', params=params)
    if resp.status_code != 200:
        print(f'  [{resp.status_code}] {resp.json().get("issue", [{}])[0].get("diagnostics", "")}')
        return
    body = resp.json()
    print(f'  Query: "{query}" (mode={mode}) → {body["total"]} results')
    for entry in body['entry']:
        r = entry['resource']
        score = entry['search']['score']
        grade = entry['search']['extension'][0]['valueCode']
        src = r['system'].split('/')[-1]
        print(f'    {score:.2f} {grade:8s} {src:15s} {r["code"]:15s} {r.get("display", "")}')

# Lexical search
show_search('diabetes', mode='lexical')

  Query: "diabetes" (mode=lexical) → 5 results
    1.00 certain  sct             399144008       Bronze Diabetes
    1.00 certain  sct             73211009        Diabetes
    1.00 certain  sct             127012008       Lipoatrophic Diabetes
    1.00 certain  icd-10-cm       E08-E13         Diabetes
    1.00 certain  sct             11530004        Brittle Diabetes


In [27]:
# Semantic search: catches novel phrasings where BM25 fails
show_search('high blood sugar', mode='semantic')
show_search('chest feels tight', mode='semantic')
show_search('water pill', mode='semantic')

  Query: "high blood sugar" (mode=semantic) → 5 results
    0.80 probable icd-10-cm       R73             Elevated Blood Glucose Level
    0.80 probable sct             80394007        Hyperglycemia
    0.79 probable icd-10-cm       R73.9           Hyperglycemia
    0.78 probable icd-10-cm       R73.0           Abnormal Glucose
    0.76 probable icd-10-cm       R73.09          Abnormal Glucose
  Query: "chest feels tight" (mode=semantic) → 5 results
    0.59 possible icd-10-cm       R07.1           Breathing Pain
    0.58 possible icd-10-cm       R07.9           Chest Pain
    0.57 possible icd-10-cm       R07.8           Pain in Throat and Chest
    0.57 possible loinc.org       70443-7         I Feel Tightness in My Chest in the Past 7 Days
    0.52 possible icd-10-cm       R09.A           Foreign Body Sensation of the Circulatory and Respiratory System
  Query: "water pill" (mode=semantic) → 5 results
    0.68 possible rxnorm          1425976         Water Ophthalmic Product
    0.6

In [28]:
# Hybrid: best of both worlds
show_search('metformin pill', mode='hybrid')

  Query: "metformin pill" (mode=hybrid) → 5 results
    1.00 certain  rxnorm          1161611         Metformin Pill
    0.89 probable rxnorm          1156197         Glyburide / Metformin Pill
    0.86 probable rxnorm          1165206         Glipizide / Metformin Pill
    0.86 probable rxnorm          1161600         Metformin / Repaglinide Pill
    0.82 probable rxnorm          1545147         Canagliflozin / Metformin Pill


---
## 7. $closure — Fast Subsumption Table

Pre-compute hierarchy relationships for O(1) subsumption checks.
Useful for validating codes in large ValueSets or patient records.

In [29]:
# Initialize a closure
resp = client.post('/fhir/CodeSystem/$closure', json={
    'resourceType': 'Parameters',
    'parameter': [{'name': 'name', 'valueString': 'demo-closure'}],
})
version = [p for p in resp.json()['parameter'] if p['name'] == 'return'][0]
print(f'Initialized closure: version={version["valueString"]}')

# Add concepts
resp = client.post('/fhir/CodeSystem/$closure', json={
    'resourceType': 'Parameters',
    'parameter': [
        {'name': 'name', 'valueString': 'demo-closure'},
        {'name': 'concept', 'valueCoding': {
            'system': 'http://snomed.info/sct', 'code': '73211009', 'display': 'Diabetes'}},
        {'name': 'concept', 'valueCoding': {
            'system': 'http://snomed.info/sct', 'code': '44054006', 'display': 'Type 2 diabetes'}},
    ],
})
version = [p for p in resp.json()['parameter'] if p['name'] == 'return'][0]
concepts = [p for p in resp.json()['parameter'] if p['name'] == 'concept']
print(f'After adding 2 concepts: version={version["valueString"]}, {len(concepts)} concepts')

# Verify subsumption via closure (O(1) lookup)
from medterm4ds.engines.fhir.closure import get_closure_manager
closure = get_closure_manager().get('demo-closure')
print(f'Closure check(73211009, 44054006): {closure.check("73211009", "44054006")}')
print(f'Closure check(44054006, 73211009): {closure.check("44054006", "73211009")}')

Initialized closure: version=b286fd708d7e


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

After adding 2 concepts: version=771d5dcd0ff2, 2 concepts
Closure check(73211009, 44054006): subsumes
Closure check(44054006, 73211009): subsumed-by


---
## Real-World Scenario: Patient Medication Reconciliation

A patient has a FHIR Condition with SNOMED code 44054006 (Type 2 diabetes).
We want to:
1. Get the patient-friendly name
2. Find the canonical ICD-10 code for association lookup
3. Search for related medications
4. Validate that a prescribed code exists

In [30]:
CONDITION_CODE = '44054006'
CONDITION_SYSTEM = 'http://snomed.info/sct'

# Step 1: Get patient-friendly name + canonical code
resp = client.get('/fhir/CodeSystem/$lookup', params={
    'system': CONDITION_SYSTEM, 'code': CONDITION_CODE,
})
body = resp.json()
display = [p for p in body['parameter'] if p['name'] == 'display'][0]['valueString']
props = {}
for p in body['parameter']:
    if p['name'] == 'property':
        parts = {pt['name']: pt for pt in p['part']}
        props[parts['code']['valueCode']] = parts.get('value', {}).get('valueString', parts.get('value', {}).get('valueCode', '?'))

print(f'Patient-friendly: {props.get("patient-friendly", display)}')
print(f'Canonical ICD-10: {props.get("canonical-code", "?")} ({props.get("canonical-system", "?")})')

# Step 2: Search for related medications
print(f'\nMedications for "{display}":')
show_search('metformin', mode='hybrid', system='http://www.nlm.nih.gov/research/umls/rxnorm')

# Step 3: Validate a prescribed RxNorm code
resp = client.get('/fhir/CodeSystem/$validate-code', params={
    'system': 'http://www.nlm.nih.gov/research/umls/rxnorm', 'code': '860975',
})
result = [p for p in resp.json()['parameter'] if p['name'] == 'result'][0]
print(f'\nPrescribed RxNorm 860975 valid: {result["valueBoolean"]}')

Patient-friendly: Diabetes Type 2
Canonical ICD-10: E11 (icd10)

Medications for "Type 2 diabetes mellitus":
  Query: "metformin" (mode=hybrid) → 5 results
    1.00 certain  sct             734558009       Metformin
    1.00 certain  sct             109083009       Metformin
    0.82 probable rxnorm          285129          Glyburide / Metformin
    0.82 probable sct             768500006       Glyburide / Metformin
    0.81 probable rxnorm          352381          Glipizide / Metformin

Prescribed RxNorm 860975 valid: True


---
## Summary

| Operation | Use case | Latency |
|---|---|---|
| `$lookup` | Display name + properties for a known code | ~5ms |
| `$validate-code` | Check if a code exists | ~5ms |
| `$translate` | Map between code systems (SNOMED ↔ ICD-10) | ~10ms |
| `$subsumes` | Hierarchy relationship check | ~10ms |
| `$expand` filter | EHR autocomplete dropdown | ~50ms |
| `$expand` intensional | All descendants of a concept | ~100ms |
| `$closure` | Pre-compute subsumption for batch checks | ~10ms |
| `$search` lexical | BM25 text search | ~1ms |
| `$search` semantic | Embedding search for novel phrasings | ~100ms |
| `$search` hybrid | Best accuracy (BM25 + re-rank) | ~110ms |

The server is FHIR R4 conformant (validated against HAPI reference server)
and deployable to HF Spaces (0.32 GB lookup DB + pre-computed JSONs).

In [31]:
# Cleanup: close the server connection
client.__exit__(None, None, None)
print('Server connection closed.')

Server connection closed.
